# Student Risk Decision Support — Data Preparation

This notebook builds a **day-28 prediction cohort** from the Open University Learning Analytics Dataset (OULAD). The unit of analysis is one student enrolled in one module presentation, identified by `code_module`, `code_presentation`, and `id_student`.

**Prediction task:** At the beginning of day 28, identify enrollments whose final result will be `Fail` or `Withdrawn`. The other outcomes, `Pass` and `Distinction`, form the comparison class.

**Observation window:** VLE activity on days 0–27. Negative dates represent activity before the presentation starts and are deliberately excluded from this first version. Registration and withdrawal dates determine whether an enrollment belongs to the day-28 cohort; neither withdrawal dates nor `final_result` may be used as model features.

This notebook runs in the Google Cloud Dataproc Jupyter environment.

## 1. Configure Spark and the data location

In [1]:
# Start Spark and configure the OULAD data path.
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

spark = SparkSession.builder.appName("StudentRiskDecisionSupport").getOrCreate()

DATA_DIR = "gs://6892bucket/student-risk/raw"
CUTOFF_DAY = 28
KEYS = ["code_module", "code_presentation", "id_student"]

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 13:57:42 INFO SparkEnv: Registering MapOutputTracker
26/10/09 13:57:42 INFO SparkEnv: Registering BlockManagerMaster
26/10/09 13:57:42 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/10/09 13:57:42 INFO SparkEnv: Registering OutputCommitCoordinator


## 2. Read the source tables

In [2]:
# Define types for the three source CSV files.
student_info_schema = StructType([
    StructField("code_module", StringType(), True),
    StructField("code_presentation", StringType(), True),
    StructField("id_student", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("region", StringType(), True),
    StructField("highest_education", StringType(), True),
    StructField("imd_band", StringType(), True),
    StructField("age_band", StringType(), True),
    StructField("num_of_prev_attempts", IntegerType(), True),
    StructField("studied_credits", IntegerType(), True),
    StructField("disability", StringType(), True),
    StructField("final_result", StringType(), True),
])

registration_schema = StructType([
    StructField("code_module", StringType(), True),
    StructField("code_presentation", StringType(), True),
    StructField("id_student", IntegerType(), True),
    StructField("date_registration", IntegerType(), True),
    StructField("date_unregistration", IntegerType(), True),
])

student_vle_schema = StructType([
    StructField("code_module", StringType(), True),
    StructField("code_presentation", StringType(), True),
    StructField("id_student", IntegerType(), True),
    StructField("id_site", IntegerType(), True),
    StructField("date", IntegerType(), True),
    StructField("sum_click", IntegerType(), True),
])


In [3]:
# Load student information, registration, and VLE events.
def read_oulad_csv(filename, schema):
    return (
        spark.read
        .option("header", True)
        .option("nullValue", "?")
        .schema(schema)
        .csv(f"{DATA_DIR}/{filename}")
    )


student_info = read_oulad_csv("studentInfo.csv", student_info_schema)
registration = read_oulad_csv("studentRegistration.csv", registration_schema)
student_vle = read_oulad_csv("studentVle.csv", student_vle_schema)

student_vle.printSchema()
student_vle.show(5, truncate=False)


root
 |-- code_module: string (nullable = true)
 |-- code_presentation: string (nullable = true)
 |-- id_student: integer (nullable = true)
 |-- id_site: integer (nullable = true)
 |-- date: integer (nullable = true)
 |-- sum_click: integer (nullable = true)



+-----------+-----------------+----------+-------+----+---------+
|code_module|code_presentation|id_student|id_site|date|sum_click|
+-----------+-----------------+----------+-------+----+---------+
|AAA        |2013J            |28400     |546652 |-10 |4        |
|AAA        |2013J            |28400     |546652 |-10 |1        |
|AAA        |2013J            |28400     |546652 |-10 |1        |
|AAA        |2013J            |28400     |546614 |-10 |11       |
|AAA        |2013J            |28400     |546714 |-10 |1        |
+-----------+-----------------+----------+-------+----+---------+
only showing top 5 rows



## 3. Audit outcomes and enrollment dates

OULAD uses `?` for some missing values. The CSV reader maps that marker to null before dates are compared with day 28.

In [4]:
# Check source counts, duplicate enrollment keys, and outcome labels.
info_count = student_info.count()
registration_count = registration.count()

info_duplicate_keys = (
    student_info.groupBy(*KEYS).count().filter(F.col("count") > 1).count()
)
registration_duplicate_keys = (
    registration.groupBy(*KEYS).count().filter(F.col("count") > 1).count()
)

print("studentInfo rows:", info_count)
print("studentRegistration rows:", registration_count)
print("Duplicate keys in studentInfo:", info_duplicate_keys)
print("Duplicate keys in studentRegistration:", registration_duplicate_keys)
student_info.groupBy("final_result").count().orderBy("final_result").show()

assert info_duplicate_keys == 0 and registration_duplicate_keys == 0


studentInfo rows: 32593
studentRegistration rows: 32593
Duplicate keys in studentInfo: 0
Duplicate keys in studentRegistration: 0
+------------+-----+
|final_result|count|
+------------+-----+
| Distinction| 3024|
|        Fail| 7052|
|        Pass|12361|
|   Withdrawn|10156|
+------------+-----+



In [5]:
# Inspect missing dates and registrations outside the prediction window.
registration.agg(
    F.count(F.when(F.col("date_registration").isNull(), 1)).alias("missing_registration_dates"),
    F.count(F.when(F.col("date_unregistration").isNull(), 1)).alias("missing_withdrawal_dates"),
    F.count(F.when(F.col("date_unregistration") < CUTOFF_DAY, 1)).alias("withdrawn_before_day_28"),
    F.count(F.when(F.col("date_registration") >= CUTOFF_DAY, 1)).alias("registered_on_or_after_day_28"),
).show()


+--------------------------+------------------------+-----------------------+-----------------------------+
|missing_registration_dates|missing_withdrawal_dates|withdrawn_before_day_28|registered_on_or_after_day_28|
+--------------------------+------------------------+-----------------------+-----------------------------+
|                        45|                   22521|                   5040|                           18|
+--------------------------+------------------------+-----------------------+-----------------------------+



## 4. Define the day-28 prediction cohort

An enrollment is eligible if registration occurred before day 28 and no withdrawal occurred before day 28. Missing registration dates cannot establish eligibility. A missing withdrawal date does not mean an early withdrawal.

In [6]:
# Join outcomes to registration and keep eligible enrollments.
cohort = student_info.join(
    registration.select(*KEYS, "date_registration", "date_unregistration"),
    on=KEYS,
    how="inner",
)

cohort_count = cohort.count()
print("Joined enrollments:", cohort_count)
assert cohort_count == info_count == registration_count

eligible = (
    cohort
    .filter(
        (F.col("date_registration") < CUTOFF_DAY)
        & (
            F.col("date_unregistration").isNull()
            | (F.col("date_unregistration") >= CUTOFF_DAY)
        )
    )
    .withColumn(
        "at_risk",
        F.when(F.col("final_result").isin("Fail", "Withdrawn"), 1).otherwise(0),
    )
)

eligible_count = eligible.count()
print("Eligible day-28 enrollments:", eligible_count)
eligible.groupBy("final_result", "at_risk").count().orderBy("final_result").show()


Joined enrollments: 32593
Eligible day-28 enrollments: 27528
+------------+-------+-----+
|final_result|at_risk|count|
+------------+-------+-----+
| Distinction|      0| 3023|
|        Fail|      1| 7032|
|        Pass|      0|12355|
|   Withdrawn|      1| 5118|
+------------+-------+-----+



## 5. Aggregate activity available by day 28

The activity window is days 0–27. Each aggregated row represents one enrollment. The three initial features measure recorded online activity; they do not measure all forms of studying.

In [7]:
# Summarize VLE clicks, active days, and unique resources.
early_vle = student_vle.filter(
    (F.col("date") >= 0) & (F.col("date") < CUTOFF_DAY)
)

activity_features = (
    early_vle
    .groupBy(*KEYS)
    .agg(
        F.sum("sum_click").alias("clicks_0_27"),
        F.countDistinct("date").alias("active_days_0_27"),
        F.countDistinct("id_site").alias("resources_0_27"),
    )
)

activity_features.show(5, truncate=False)
print("Enrollments with recorded day-0-to-27 activity:", activity_features.count())


+-----------+-----------------+----------+-----------+----------------+--------------+
|code_module|code_presentation|id_student|clicks_0_27|active_days_0_27|resources_0_27|
+-----------+-----------------+----------+-----------+----------------+--------------+
|BBB        |2013B            |558689    |187        |9               |14            |
|BBB        |2013J            |593299    |174        |10              |21            |
|BBB        |2014B            |408469    |136        |10              |17            |
|BBB        |2014B            |445136    |36         |3               |8             |
|FFF        |2013B            |515844    |281        |14              |33            |
+-----------+-----------------+----------+-----------+----------------+--------------+
only showing top 5 rows



Enrollments with recorded day-0-to-27 activity: 27886


## 6. Join activity to the prediction cohort

Use a left join to retain eligible enrollments with no VLE events during the window. Count missing activity rows before setting these three aggregated features to zero. Remove registration dates after cohort selection to prevent accidental use of withdrawal timing in modeling.

In [8]:
# Keep inactive enrollments and fill only the activity features.
joined = eligible.join(activity_features, on=KEYS, how="left")

joined_count = joined.count()
no_recorded_activity = joined.filter(F.col("clicks_0_27").isNull()).count()
print("Rows after the activity join:", joined_count)
print("Eligible enrollments without recorded VLE activity:", no_recorded_activity)
assert joined_count == eligible_count

ACTIVITY_FEATURES = ["clicks_0_27", "active_days_0_27", "resources_0_27"]
model_data = (
    joined
    .fillna(0, subset=ACTIVITY_FEATURES)
    .drop("date_registration", "date_unregistration")
    .cache()
)

print("Prepared rows:", model_data.count())
model_data.groupBy("at_risk").count().orderBy("at_risk").show()


Rows after the activity join: 27528
Eligible enrollments without recorded VLE activity: 1224


26/10/09 13:58:08 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


Prepared rows: 27528


+-------+-----+
|at_risk|count|
+-------+-----+
|      0|15378|
|      1|12150|
+-------+-----+



## 7. Audit module presentations and choose a holdout

Keep all `2014J` presentations for the final test set and use earlier presentations for training. This checks performance on a later presentation rather than mixing enrollment rows from the same presentation across a random split. `final_result` remains only for auditing and is dropped from the split datasets below.

In [9]:
# Compare sample sizes and risk rates across 22 presentations.
presentation_summary = (
    model_data
    .groupBy("code_module", "code_presentation")
    .agg(
        F.count("*").alias("enrollments"),
        F.sum("at_risk").alias("at_risk_count"),
        F.round(F.avg("at_risk"), 3).alias("risk_rate"),
    )
    .orderBy("code_module", "code_presentation")
)

presentation_summary.show(30, truncate=False)


+-----------+-----------------+-----------+-------------+---------+
|code_module|code_presentation|enrollments|at_risk_count|risk_rate|
+-----------+-----------------+-----------+-------------+---------+
|AAA        |2013J            |369        |91           |0.247    |
|AAA        |2014J            |348        |95           |0.273    |
|BBB        |2013B            |1529       |727          |0.475    |
|BBB        |2013J            |1834       |763          |0.416    |
|BBB        |2014B            |1280       |553          |0.432    |
|BBB        |2014J            |1786       |635          |0.356    |
|CCC        |2014B            |1500       |837          |0.558    |
|CCC        |2014J            |2020       |1006         |0.498    |
|DDD        |2013B            |1141       |632          |0.554    |
|DDD        |2013J            |1647       |819          |0.497    |
|DDD        |2014B            |1028       |550          |0.535    |
|DDD        |2014J            |1463       |671  

In [10]:
# Separate earlier presentations from the 2014J holdout.
train_cohort = (
    model_data
    .filter(F.col("code_presentation") != "2014J")
    .drop("final_result")
)
test_cohort = (
    model_data
    .filter(F.col("code_presentation") == "2014J")
    .drop("final_result")
)

train_count = train_cohort.count()
test_count = test_cohort.count()
print("Training enrollments:", train_count)
print("Holdout enrollments:", test_count)
assert train_count + test_count == eligible_count

print("Training labels:")
train_cohort.groupBy("at_risk").count().orderBy("at_risk").show()
print("Holdout labels:")
test_cohort.groupBy("at_risk").count().orderBy("at_risk").show()

BASELINE_FEATURES = ACTIVITY_FEATURES
print("Initial model features:", BASELINE_FEATURES)


Training enrollments: 18315
Holdout enrollments: 9213
Training labels:


+-------+-----+
|at_risk|count|
+-------+-----+
|      0| 9923|
|      1| 8392|
+-------+-----+

Holdout labels:


+-------+-----+
|at_risk|count|
+-------+-----+
|      0| 5455|
|      1| 3758|
+-------+-----+

Initial model features: ['clicks_0_27', 'active_days_0_27', 'resources_0_27']


In [11]:
# Save the prepared training and holdout cohorts.
PROCESSED_DIR = "gs://6892bucket/student-risk/processed/day28_v1"

train_cohort.write.mode("overwrite").parquet(f"{PROCESSED_DIR}/train")
test_cohort.write.mode("overwrite").parquet(f"{PROCESSED_DIR}/holdout")